<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:200%;text-align:center;border-radius:15px;padding:20px;">🐦 BirdCLEF+ 2026: Complete EDA, Audio Analysis & Preprocessing</p>

<p style="background-color:#1a472a;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;padding:10px;">📘 PART 1 — EDA & Feature Engineering</p>

# 🌿 Welcome to BirdCLEF+ 2026!

The **BirdCLEF+ 2026** competition challenges us to identify bird species from audio recordings in the Pantanal, South America, the world's largest tropical wetland and home to 650+ bird species.

## 📖 What You'll Learn in This Notebook (Part 1)
- 🔍 **Comprehensive EDA** of the metadata: species distribution, geography, temporal patterns, quality
- 🎵 **Audio Deep Dive** — waveforms, STFT spectrograms, mel-spectrograms, MFCCs
- 🔀 **Cross-species comparison** of acoustic signatures
- ⚙️ **Feature extraction pipeline** (reusable `AudioFeatureExtractor` class)
- 🎛️ **Data augmentation** strategies (time-shift, noise, pitch-shift, time-stretch, mixup)

> 🚀 **Part 2** (training + inference + submission) is a separate notebook: **"BirdCLEF+ 2026: PyTorch Baseline Training & Inference"**

<p style="background-color:#1a472a;font-family:newtimeroman;color:#FFF9ED;font-size:140%;text-align:center;border-radius:10px;padding:15px;">📑 TABLE OF CONTENTS</p>

1. [📚 Libraries & Setup](#libraries)
2. [📂 Dataset Overview & Metadata](#metadata)
3. [🐦 Species Distribution & Class Imbalance](#species-dist)
4. [🌍 Geographic Distribution](#geography)
5. [⭐ Recording Quality Analysis](#quality)
6. [📅 Temporal Patterns](#temporal)
7. [🎵 Audio Deep Dive](#audio-dive)
8. [🎼 Mel-Spectrograms & MFCCs](#mel-mfcc)
9. [🔀 Cross-Species Audio Comparison](#cross-species)
10. [⚙️ Feature Engineering Pipeline](#feature-eng)
11. [🎛️ Data Augmentation Strategies](#augmentation)
12. [🎯 Next Steps → Part 2: Training Notebook](#next-steps)


<a id="libraries"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">📚 LIBRARIES & SETUP</p>

In [ ]:
# Core data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Audio processing
import librosa
import librosa.display
from librosa.feature import melspectrogram, mfcc

# Audio playback
from IPython.display import Audio, display

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

# Utilities
import os
import warnings
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import json

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✅ All libraries imported successfully!")
print(f"PyTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

In [ ]:
import os

DATA_DIR = '/kaggle/input/competitions/birdclef-2026'
TRAIN_AUDIO_DIR = os.path.join(DATA_DIR, 'train_audio')
TEST_AUDIO_DIR = os.path.join(DATA_DIR, 'test_soundscapes')

# Prioritize train.csv over soundscape labels - train.csv is the main metadata
metadata_priority = ['train.csv', 'train_metadata.csv', 'metadata.csv']
METADATA_PATH = None
for fname in metadata_priority:
    candidate = os.path.join(DATA_DIR, fname)
    if os.path.exists(candidate):
        METADATA_PATH = candidate
        break

SOUNDSCAPE_LABELS_PATH = os.path.join(DATA_DIR, 'train_soundscapes_labels.csv')
TAXONOMY_PATH = os.path.join(DATA_DIR, 'taxonomy.csv')

print(f"✅ Paths resolved:")
print(f"   METADATA_PATH:          {METADATA_PATH}")
print(f"   SOUNDSCAPE_LABELS_PATH: {SOUNDSCAPE_LABELS_PATH}")
print(f"   TAXONOMY_PATH:          {TAXONOMY_PATH}")
print(f"   TRAIN_AUDIO_DIR:        {TRAIN_AUDIO_DIR}")
print(f"   TEST_AUDIO_DIR:         {TEST_AUDIO_DIR}")

# Show all CSVs available
print(f"\n📄 All CSV files in dataset:")
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith('.csv'):
        fullpath = os.path.join(DATA_DIR, f)
        size_kb = os.path.getsize(fullpath) / 1024
        print(f"   {f} ({size_kb:.1f} KB)")

<a id="metadata"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">🔍 LOAD & EXPLORE METADATA</p>

In [ ]:
# Load the main training metadata (train.csv, NOT soundscape labels)
try:
    df = pd.read_csv(METADATA_PATH)
    print(f"📊 Dataset shape: {df.shape}")
    print(f"\n📋 Columns: {list(df.columns)}")
    print(f"\n🔎 First few rows:")
    display(df.head())
except FileNotFoundError:
    print(f"❌ Metadata not found at {METADATA_PATH}")
    print("Please ensure the BirdCLEF+ 2026 competition dataset is added to your notebook.")
    raise
except Exception as e:
    print(f"❌ Error loading metadata: {e}")
    raise

In [ ]:
# Detailed metadata exploration
print("📈 Dataset Info:")
print(f"Total recordings: {len(df)}")
print(f"Unique species: {df['primary_label'].nunique()}")
print(f"\n❌ Missing values:")
print(df.isnull().sum())
print(f"\n📊 Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Display sample metadata
print("Sample recording metadata:")
sample_idx = 0
for col in df.columns:
    print(f"{col}: {df.iloc[sample_idx][col]}")

📌 **Note**: The metadata contains geographical coordinates (latitude/longitude) and quality ratings that are crucial for understanding data bias and quality considerations in model development.

<a id="species-dist"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">🐦 EDA PART 1: SPECIES DISTRIBUTION</p>

In [ ]:
# Species distribution
species_counts = df['primary_label'].value_counts().reset_index()
species_counts.columns = ['species', 'count']

print(f"🐦 Total species: {len(species_counts)}")
print(f"\n📊 Class imbalance:")
print(f"Max recordings (single species): {species_counts['count'].max()}")
print(f"Min recordings (single species): {species_counts['count'].min()}")
print(f"Mean recordings per species: {species_counts['count'].mean():.1f}")
print(f"Median recordings per species: {species_counts['count'].median():.1f}")

print(f"\n🔝 Top 15 most recorded species:")
print(species_counts.head(15).to_string(index=False))

In [ ]:
# Visualize top species
fig = px.bar(
    species_counts.head(20),
    x='species',
    y='count',
    title='Top 20 Most Recorded Bird Species',
    labels={'species': 'Species', 'count': 'Number of Recordings'},
    color='count',
    color_continuous_scale='Greens'
)
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

print("\n📊 Species distribution visualization created.")

In [ ]:
# Class imbalance analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of recording counts
axes[0].hist(species_counts['count'], bins=50, color='#2E8B57', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Recordings per Species', fontsize=12)
axes[0].set_ylabel('Number of Species', fontsize=12)
axes[0].set_title('Distribution of Recordings Across Species', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Log scale histogram
axes[1].hist(species_counts['count'], bins=50, color='#1a472a', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Recordings per Species', fontsize=12)
axes[1].set_ylabel('Number of Species (log scale)', fontsize=12)
axes[1].set_title('Distribution (Log Scale)', fontsize=13, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📌 **Note**: Significant class imbalance detected. This will require careful consideration during model training (weighted loss, oversampling, or focal loss).")

In [ ]:
# Recording type distribution (defensive: 'type' column may not exist)
if 'type' in df.columns:
    recording_type_dist = df['type'].value_counts().head(15)
    print("🎙️ Recording types:")
    print(recording_type_dist)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors_list = plt.cm.Greens(np.linspace(0.4, 0.9, len(recording_type_dist)))
    recording_type_dist.plot(kind='barh', color=colors_list, ax=ax)
    ax.set_xlabel('Count', fontsize=12)
    ax.set_title('Distribution of Recording Types (Top 15)', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ 'type' column not in metadata. Available columns:", list(df.columns))

<a id="geography"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">🗺️ EDA PART 2: GEOGRAPHIC ANALYSIS</p>

In [ ]:
# Geographic analysis
print("📍 Geographic Distribution:")
print(f"Latitude range: {df['latitude'].min():.2f} to {df['latitude'].max():.2f}")
print(f"Longitude range: {df['longitude'].min():.2f} to {df['longitude'].max():.2f}")
print(f"Recordings with valid coordinates: {df[['latitude', 'longitude']].notna().all(axis=1).sum()}")

# Get unique recording locations
df_with_coords = df[df[['latitude', 'longitude']].notna().all(axis=1)].copy()
print(f"\n🗺️ Unique recording locations: {len(df_with_coords[['latitude', 'longitude']].drop_duplicates())}")

In [ ]:
# Scatter plot of recording locations
fig = px.scatter_geo(
    df_with_coords,
    lat='latitude',
    lon='longitude',
    hover_name='primary_label',
    hover_data={'latitude': ':.2f', 'longitude': ':.2f'},
    title='Geographic Distribution of BirdCLEF+ 2026 Recordings',
    projection='natural earth',
    color_discrete_sequence=['#2E8B57']
)
fig.update_geos(scope='south america')
fig.show()

print("\n🗺️ Interactive map showing recording locations in South America.")

In [ ]:
# Detailed geographic scatter plot
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(df_with_coords['longitude'], df_with_coords['latitude'], 
                     c=df_with_coords['rating'], cmap='Greens', s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Recording Locations Colored by Quality Rating', fontsize=13, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Recording Quality Rating', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Geographic clustering - which species dominate which regions?
# Create hexbin plot
fig, ax = plt.subplots(figsize=(12, 8))
hexbin = ax.hexbin(df_with_coords['longitude'], df_with_coords['latitude'], 
                    gridsize=20, cmap='YlGn', mincnt=1)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Recording Density by Location (Hexbin)', fontsize=13, fontweight='bold')
cbar = plt.colorbar(hexbin, ax=ax)
cbar.set_label('Number of Recordings', fontsize=11)
plt.tight_layout()
plt.show()

print("\n📌 **Note**: Geographic distribution reveals potential sampling bias. Some regions may have more recordings than others.")

<a id="quality"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">🎙️ EDA PART 3: RECORDING QUALITY</p>

In [ ]:
# Recording quality analysis
print("⭐ Recording Quality Ratings:")
print(df['rating'].describe())
print(f"\nRating distribution:")
print(df['rating'].value_counts().sort_index())

# License distribution
print(f"\n📜 License distribution:")
print(df['license'].value_counts())

In [ ]:
# ⭐ Improved Quality Ratings Dashboard (clean + readable + professional)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("⭐ Recording Quality Insights", fontsize=16, fontweight="bold", y=1.02)

# -------------------------------------------------------
# 1. Histogram of ratings
# -------------------------------------------------------
sns.histplot(
    df["rating"].dropna(),
    bins=20,
    kde=True,
    color="#2E8B57",
    edgecolor="black",
    ax=axes[0, 0]
)

axes[0, 0].set_title("Distribution of Quality Ratings", fontweight="bold")
axes[0, 0].set_xlabel("Rating")
axes[0, 0].set_ylabel("Frequency")

# -------------------------------------------------------
# 2. Boxplot by Top Recording Types only
# -------------------------------------------------------
top_types = df["type"].astype(str).value_counts().head(10).index

subset = df[df["type"].astype(str).isin(top_types)]

sns.boxplot(
    data=subset,
    x="type",
    y="rating",
    palette="Greens",
    ax=axes[0, 1]
)

axes[0, 1].set_title("Ratings by Top Recording Types", fontweight="bold")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("Rating")
axes[0, 1].tick_params(axis="x", rotation=45)

# -------------------------------------------------------
# 3. License Distribution
# -------------------------------------------------------
license_counts = df["license"].value_counts().head(8)

sns.barplot(
    x=license_counts.index,
    y=license_counts.values,
    palette="viridis",
    ax=axes[1, 0]
)

axes[1, 0].set_title("License Distribution", fontweight="bold")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("Count")
axes[1, 0].tick_params(axis="x", rotation=45)

# -------------------------------------------------------
# 4. Average Rating by Top Recording Types
# -------------------------------------------------------
type_rating = (
    subset.groupby("type")["rating"]
    .mean()
    .sort_values(ascending=False)
)

sns.barplot(
    x=type_rating.index,
    y=type_rating.values,
    palette="crest",
    ax=axes[1, 1]
)

axes[1, 1].set_title("Average Rating by Recording Type", fontweight="bold")
axes[1, 1].set_xlabel("")
axes[1, 1].set_ylabel("Mean Rating")
axes[1, 1].tick_params(axis="x", rotation=45)

# -------------------------------------------------------
plt.tight_layout()
plt.show()

In [ ]:
# Analyze relationship between quality and species popularity
species_quality = df.groupby('primary_label').agg({
    'rating': ['mean', 'count'],
    'filename': 'count'
}).reset_index()
species_quality.columns = ['species', 'avg_rating', 'count', 'total']
species_quality = species_quality.sort_values('count', ascending=False)

fig = px.scatter(
    species_quality.head(50),
    x='count',
    y='avg_rating',
    hover_name='species',
    size='count',
    color='avg_rating',
    title='Species Popularity vs Average Recording Quality (Top 50 Species)',
    labels={'count': 'Number of Recordings', 'avg_rating': 'Average Quality Rating'},
    color_continuous_scale='Greens'
)
fig.show()

print("\n📌 **Note**: Quality varies across species. Some popular species have lower average ratings, which could affect model training.")

<a id="temporal"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">📅 EDA PART 4: TEMPORAL PATTERNS</p>

In [ ]:
# Parse dates if available
# Check which columns might contain date/time information
print("Checking for temporal columns:")
temporal_cols = ['date', 'datetime', 'year', 'month', 'day', 'time', 'hour']
for col in df.columns:
    if any(t in col.lower() for t in temporal_cols):
        print(f"Found: {col}")
        print(df[col].head())
        print()

In [ ]:
# If there's a date column, extract temporal features
if 'recorded_date' in df.columns or 'date' in df.columns:
    date_col = 'recorded_date' if 'recorded_date' in df.columns else 'date'
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    
    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['hour'] = df[date_col].dt.hour
    df['day_of_year'] = df[date_col].dt.dayofyear
    
    # Temporal distribution
    print("📅 Temporal Distribution:")
    print(f"Recording years: {sorted(df['year'].dropna().unique())}")
    print(f"\nRecordings by year:")
    print(df['year'].value_counts().sort_index())
    

In [ ]:
# Temporal visualizations
if 'year' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # By year
    yearly = df['year'].value_counts().sort_index()
    axes[0, 0].bar(yearly.index, yearly.values, color='#2E8B57', alpha=0.7, edgecolor='black')
    axes[0, 0].set_xlabel('Year', fontsize=11)
    axes[0, 0].set_ylabel('Count', fontsize=11)
    axes[0, 0].set_title('Recordings by Year', fontsize=12, fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # By month
    monthly = df['month'].value_counts().sort_index()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    axes[0, 1].bar(monthly.index, monthly.values, color='#1a472a', alpha=0.7, edgecolor='black')
    axes[0, 1].set_xlabel('Month', fontsize=11)
    axes[0, 1].set_ylabel('Count', fontsize=11)
    axes[0, 1].set_title('Recordings by Month', fontsize=12, fontweight='bold')
    axes[0, 1].set_xticks(range(1, 13))
    axes[0, 1].set_xticklabels(month_names, rotation=45)
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # By hour
    hourly = df['hour'].value_counts().sort_index()
    axes[1, 0].plot(hourly.index, hourly.values, marker='o', color='#228B22', linewidth=2, markersize=6)
    axes[1, 0].fill_between(hourly.index, hourly.values, alpha=0.3, color='#228B22')
    axes[1, 0].set_xlabel('Hour of Day', fontsize=11)
    axes[1, 0].set_ylabel('Count', fontsize=11)
    axes[1, 0].set_title('Recordings by Hour of Day', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Day of year
    axes[1, 1].hist(df['day_of_year'].dropna(), bins=365, color='#2E8B57', alpha=0.7, edgecolor='none')
    axes[1, 1].set_xlabel('Day of Year', fontsize=11)
    axes[1, 1].set_ylabel('Count', fontsize=11)
    axes[1, 1].set_title('Seasonal Distribution of Recordings', fontsize=12, fontweight='bold')
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📌 **Note**: Bird activity varies by time of day and season. Most birds are most vocal during dawn and dusk.")

<a id="audio-dive"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">🔊 AUDIO DEEP DIVE: WAVEFORMS & SPECTROGRAMS</p>

In [ ]:
# Function to safely load audio
def load_audio_safe(audio_path, sr=32000):
    """Safely load audio file with error handling."""
    try:
        if not os.path.exists(audio_path):
            return None, None
        y, sr_actual = librosa.load(audio_path, sr=sr)
        return y, sr_actual
    except Exception as e:
        print(f"Error loading {audio_path}: {e}")
        return None, None

# Find a sample audio file to analyze
print("🔍 Looking for sample audio files...")
sample_audio = None

# Try to find a real audio file from the dataset
if os.path.exists(TRAIN_AUDIO_DIR):
    for idx, row in df.head(100).iterrows():
        # Audio files are in numbered subdirectories like 1161364/iNat1114648.ogg
        audio_path = os.path.join(TRAIN_AUDIO_DIR, row['filename'])
        if os.path.exists(audio_path):
            sample_audio = (audio_path, row['primary_label'], row['filename'])
            print(f"✅ Found sample: {row['primary_label']} - {row['filename']}")
            break

if sample_audio is None:
    print("⚠️ No audio files found in expected directory structure.")
    print("Creating synthetic audio for demonstration...")
    # Create synthetic audio for demonstration
    sr = 32000
    duration = 5
    t = np.linspace(0, duration, int(sr * duration))
    # Synthetic bird-like sound (chirp with multiple frequencies)
    y = np.sin(2 * np.pi * 1000 * t) * np.exp(-0.5 * t)
    y += 0.5 * np.sin(2 * np.pi * 2000 * t) * np.exp(-0.3 * t)
    y += 0.1 * np.random.randn(len(t))  # Add noise
    sample_audio = (None, 'Synthetic_Bird', 'synthetic.wav')
else:
    y, sr = load_audio_safe(sample_audio[0])


In [ ]:
# Load and visualize sample audio
if sample_audio[0] is None:
    # Use synthetic audio
    sr = 32000
    duration = 5
    t = np.linspace(0, duration, int(sr * duration))
    y = np.sin(2 * np.pi * 1000 * t) * np.exp(-0.5 * t)
    y += 0.5 * np.sin(2 * np.pi * 2000 * t) * np.exp(-0.3 * t)
    y += 0.1 * np.random.randn(len(t))
    y = y / np.max(np.abs(y))  # Normalize
else:
    y, sr = load_audio_safe(sample_audio[0])

if y is not None:
    print(f"🔊 Audio loaded: {sample_audio[1]}")
    print(f"Sample rate: {sr} Hz")
    print(f"Duration: {len(y) / sr:.2f} seconds")
    print(f"Shape: {y.shape}")
    
    # Waveform visualization
    fig, ax = plt.subplots(figsize=(14, 5))
    time_axis = np.arange(len(y)) / sr
    ax.plot(time_axis, y, color='#2E8B57', linewidth=0.8, alpha=0.8)
    ax.fill_between(time_axis, y, alpha=0.3, color='#2E8B57')
    ax.set_xlabel('Time (seconds)', fontsize=12)
    ax.set_ylabel('Amplitude', fontsize=12)
    ax.set_title(f'Waveform: {sample_audio[1]} - {sample_audio[2]}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Play audio
    print("\n🎵 Audio playback:")
    display(Audio(data=y, rate=sr))

In [ ]:
# Spectrogram (STFT) visualization
if y is not None:
    # Compute STFT
    D = librosa.stft(y, n_fft=2048, hop_length=512)
    S_db = librosa.power_to_db(np.abs(D) ** 2, ref=np.max)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=512,
        x_axis='time',
        y_axis='log',
        cmap='viridis',
        ax=ax
    )
    ax.set_title(f'Log-Frequency Spectrogram (STFT)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Frequency (Hz)', fontsize=12)
    ax.set_xlabel('Time (s)', fontsize=12)
    cbar = fig.colorbar(img, ax=ax, format='%+2.0f dB')
    cbar.set_label('Power (dB)', fontsize=11)
    plt.tight_layout()
    plt.show()
    
    print("✅ Spectrogram computed from Short-Time Fourier Transform (STFT)")

<a id="mel-mfcc"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">🎵 MEL-SPECTROGRAMS & MFCCs</p>

In [ ]:
# Mel-spectrogram computation
if y is not None:
    # Parameters for mel-spectrogram
    n_mels = 128
    fmax = sr // 2  # Nyquist frequency
    
    # Compute mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=2048,
        hop_length=512,
        n_mels=n_mels,
        fmax=fmax
    )
    
    # Convert to log scale
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    print(f"🎵 Mel-spectrogram shape: {mel_spec_db.shape}")
    print(f"Frequency bins: {mel_spec_db.shape[0]}")
    print(f"Time steps: {mel_spec_db.shape[1]}")
    
    # Visualize mel-spectrogram
    fig, ax = plt.subplots(figsize=(14, 6))
    img = librosa.display.specshow(
        mel_spec_db,
        sr=sr,
        hop_length=512,
        x_axis='time',
        y_axis='mel',
        fmax=fmax,
        cmap='magma',
        ax=ax
    )
    ax.set_title('Mel-Spectrogram (128 mel-bands)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Mel-Frequency', fontsize=12)
    ax.set_xlabel('Time (s)', fontsize=12)
    cbar = fig.colorbar(img, ax=ax, format='%+2.0f dB')
    cbar.set_label('Power (dB)', fontsize=11)
    plt.tight_layout()
    plt.show()
    
    print("\n📌 **Note**: Mel-spectrograms are better aligned with human auditory perception and are widely used in bird audio classification.")

In [ ]:
# MFCC (Mel-Frequency Cepstral Coefficients) computation
if y is not None:
    # Compute MFCCs
    n_mfcc = 13
    mfcc_features = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=2048,
        hop_length=512
    )
    
    print(f"🎼 MFCC shape: {mfcc_features.shape}")
    print(f"Number of coefficients: {mfcc_features.shape[0]}")
    print(f"Time steps: {mfcc_features.shape[1]}")
    
    # Visualize MFCCs
    fig, ax = plt.subplots(figsize=(14, 6))
    img = librosa.display.specshow(
        mfcc_features,
        sr=sr,
        hop_length=512,
        x_axis='time',
        cmap='coolwarm',
        ax=ax
    )
    ax.set_ylabel('MFCC Coefficient', fontsize=12)
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_title('MFCC Features (13 coefficients)', fontsize=13, fontweight='bold')
    cbar = fig.colorbar(img, ax=ax)
    cbar.set_label('Value', fontsize=11)
    plt.tight_layout()
    plt.show()
    
    print("\n✅ MFCCs computed - representing spectral characteristics as perceived by human ear.")

<a id="cross-species"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">🎼 CROSS-SPECIES AUDIO COMPARISON</p>

In [ ]:
# Load multiple sample audio files from different species
print("🔍 Loading sample recordings from different species...")
species_samples = {}
target_species = df['primary_label'].value_counts().head(5).index.tolist()

for species in target_species:
    species_df = df[df['primary_label'] == species]
    for idx, row in species_df.head(20).iterrows():
        # Audio files are in numbered subdirectories, filename contains the relative path
        audio_path = os.path.join(TRAIN_AUDIO_DIR, row['filename'])
        if os.path.exists(audio_path):
            y, sr_loaded = load_audio_safe(audio_path, sr=32000)
            if y is not None:
                species_samples[species] = (y, sr_loaded, row['filename'])
                print(f"✅ Loaded {species}: {row['filename']}")
                break

if len(species_samples) == 0:
    print("⚠️ No audio files found. Creating synthetic samples for 5 species...")
    sr = 32000
    for i, species in enumerate(target_species[:5]):
        duration = 5
        t = np.linspace(0, duration, int(sr * duration))
        # Create unique synthetic audio for each species
        base_freq = 500 + i * 300
        y = np.sin(2 * np.pi * base_freq * t) * np.exp(-0.5 * t)
        y += 0.3 * np.sin(2 * np.pi * (base_freq * 1.5) * t) * np.exp(-0.4 * t)
        y += 0.1 * np.random.randn(len(t))
        y = y / np.max(np.abs(y))
        species_samples[species] = (y, sr, f'synthetic_{species}.wav')
        print(f"✅ Created synthetic: {species}")


In [ ]:
# Compare mel-spectrograms across species
if len(species_samples) > 0:
    n_species = min(4, len(species_samples))
    species_list = list(species_samples.keys())[:n_species]
    
    fig, axes = plt.subplots(n_species, 1, figsize=(14, 4 * n_species))
    if n_species == 1:
        axes = [axes]
    
    for idx, species in enumerate(species_list):
        y, sr_sp, fname = species_samples[species]
        
        # Compute mel-spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr_sp,
            n_fft=2048,
            hop_length=512,
            n_mels=128,
            fmax=sr_sp // 2
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Plot
        img = librosa.display.specshow(
            mel_spec_db,
            sr=sr_sp,
            hop_length=512,
            x_axis='time',
            y_axis='mel',
            ax=axes[idx],
            cmap='viridis'
        )
        axes[idx].set_title(f'Species: {species}', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Mel-Frequency', fontsize=11)
        fig.colorbar(img, ax=axes[idx], format='%+2.0f dB')
    
    axes[-1].set_xlabel('Time (s)', fontsize=11)
    plt.tight_layout()
    plt.show()
    
    print("\n📌 **Note**: Different bird species produce distinctly different spectral patterns, which is the basis for our classification approach.")

<a id="feature-eng"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;">⚙️ FEATURE ENGINEERING PIPELINE</p>

In [ ]:
# Define audio feature extraction class
class AudioFeatureExtractor:
    """Extract mel-spectrogram features from audio files."""
    
    def __init__(self, sr=32000, n_mels=128, n_fft=2048, hop_length=512):
        self.sr = sr
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
    
    def extract_mel_spectrogram(self, y):
        """Extract mel-spectrogram features."""
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=self.sr,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            n_mels=self.n_mels,
            fmax=self.sr // 2
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        return mel_spec_db
    
    def extract_mfcc(self, y, n_mfcc=13):
        """Extract MFCC features."""
        mfcc = librosa.feature.mfcc(
            y=y,
            sr=self.sr,
            n_mfcc=n_mfcc,
            n_fft=self.n_fft,
            hop_length=self.hop_length
        )
        return mfcc
    
    def extract_features(self, y):
        """Extract all features."""
        mel_spec = self.extract_mel_spectrogram(y)
        mfcc = self.extract_mfcc(y)
        return {
            'mel_spectrogram': mel_spec,
            'mfcc': mfcc,
            'mel_shape': mel_spec.shape,
            'mfcc_shape': mfcc.shape
        }

print("✅ AudioFeatureExtractor class defined.")

In [ ]:
# Test the feature extractor
if y is not None:
    extractor = AudioFeatureExtractor(sr=32000, n_mels=128)
    features = extractor.extract_features(y[:32000 * 5])  # Use first 5 seconds
    
    print("🎵 Feature Extraction Results:")
    print(f"Mel-spectrogram shape: {features['mel_shape']}")
    print(f"MFCC shape: {features['mfcc_shape']}")
    print("\n✅ Features extracted successfully!")

In [ ]:
# Data augmentation strategies
class AudioAugmenter:
    """Apply augmentation techniques to audio data."""
    
    @staticmethod
    def time_shift(y, shift_max=0.2):
        """Randomly shift audio in time."""
        shift = int(np.random.uniform(-shift_max, shift_max) * len(y))
        return np.roll(y, shift)
    
    @staticmethod
    def add_gaussian_noise(y, noise_factor=0.005):
        """Add Gaussian noise to audio."""
        noise = np.random.randn(len(y))
        return y + noise_factor * noise
    
    @staticmethod
    def pitch_shift(y, sr, n_steps=2):
        """Shift pitch of audio."""
        return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
    
    @staticmethod
    def time_stretch(y, rate=1.1):
        """Time stretch audio."""
        return librosa.effects.time_stretch(y, rate=rate)
    
    @staticmethod
    def mixup(y1, y2, alpha=0.5):
        """Mix two audio signals."""
        min_len = min(len(y1), len(y2))
        mixed = alpha * y1[:min_len] + (1 - alpha) * y2[:min_len]
        return mixed

print("✅ AudioAugmenter class defined with 5 augmentation techniques.")

In [ ]:
# Demonstrate augmentations
if len(species_samples) > 1:
    augmenter = AudioAugmenter()
    
    # Get two different species samples
    species1, species2 = list(species_samples.keys())[:2]
    y1, sr1, _ = species_samples[species1]
    y2, sr2, _ = species_samples[species2]
    
    # Apply augmentations
    y1_shifted = augmenter.time_shift(y1, shift_max=0.1)
    y1_noisy = augmenter.add_gaussian_noise(y1, noise_factor=0.01)
    
    # Visualize augmented waveforms
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    time_axis = np.arange(min(len(y1), 5 * sr1)) / sr1
    
    axes[0].plot(time_axis, y1[:len(time_axis)], color='#2E8B57', linewidth=0.8)
    axes[0].set_title(f'Original: {species1}', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(time_axis, y1_shifted[:len(time_axis)], color='#1a472a', linewidth=0.8)
    axes[1].set_title(f'Time-Shifted (10% max shift)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(time_axis, y1_noisy[:len(time_axis)], color='#228B22', linewidth=0.8)
    axes[2].set_title(f'Gaussian Noise Added (factor=0.01)', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('Amplitude', fontsize=11)
    axes[2].set_xlabel('Time (s)', fontsize=11)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📌 Note: Data augmentation helps prevent overfitting and makes models robust to audio variations in the real world.")

<a id="next-steps"></a>
<p style="background-color:#2E8B57;font-family:newtimeroman;color:#FFF9ED;font-size:130%;text-align:center;border-radius:10px;padding:10px;">🎯 NEXT STEPS PART 2: TRAINING & INFERENCE</p>

## Summary of Part 1 (This Notebook)

## 🔑 Final Key Findings

1. **🐦 206 Animal Sound Classes**
Includes birds, insects, mammals, reptiles, and amphibians with strong class imbalance (1 to 499 samples).

2. **🌍 Domain Shift Challenge**
Training clips are globally sourced clean recordings, while test data consists of noisy Pantanal soundscapes.

3. **⭐ Recording Quality Variation**
Ratings are highly skewed with many unrated clips and many high-quality recordings.

4. **🎵 Audio Features Are Informative**
Waveforms, spectrograms, mel-spectrograms, and MFCCs show distinct species patterns suitable for deep learning.

5. **⚙️ Modeling Ready Pipeline**
Reusable preprocessing and augmentation pipeline built for robust training.



Good luck in the competition! 🐦🎵
